# Module 5 — From Prompts to Agents

**NetOps Co. · CELL-031A · ~45 minutes**

Everything so far has been one call, one answer. An agent is different: **its next action depends
on the result of its last one**, decided by the model rather than hard-coded by you.

We build the smallest possible version by hand — no framework — so the mechanics stay visible.
When the libraries change next year, this will still make sense.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os
sys.path.append('/content/netops-genai-course/data')

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 5 — ReAct loop')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. The loop: perceive → reason → act → observe

Run the real agent from the repo and watch it work. It gets three read-only tools and a question.

In [ ]:
sys.path.append('/content/netops-genai-course/module05-react-loop')
from react_agent import run_react_agent

answer = run_react_agent('Why is CELL-031A underperforming this morning?')
print(answer)

---
## 2. The same run, rendered as a trace

Text scrolls past. A trace you can read is how you actually debug an agent — and it is exactly
what Module 10 turns into tracing and evaluation.

Notice the shape of the reasoning: it checks KPIs, then topology, *then* concludes. That order
is not an accident — it comes from the 4-layer diagnostic order in the system prompt.

In [ ]:
import re, nb_viz
from IPython.display import HTML, display
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    run_react_agent('Why is CELL-031A underperforming this morning?')
log = buf.getvalue()

steps = []
for block in re.split(r'(?=Thought:)', log):
    t = re.search(r'Thought:\s*(.+)', block)
    a = re.search(r'Action:\s*(.+)', block)
    o = re.search(r'Observation:\s*(.+)', block)
    if t or a:
        steps.append({'thought': t.group(1).strip() if t else '',
                      'action':  a.group(1).strip() if a else '',
                      'observation': o.group(1).strip() if o else ''})
display(HTML(nb_viz.trace(steps, caption='ReAct trace — CELL-031A')))

---
## 3. The circuit breakers — and what happens without them

Two guards in that loop look like defensive plumbing. They are not optional.

- **`max_steps`** — a hard ceiling on how many times round the loop
- **`call_history`** — cycle detection on `(tool, argument)` pairs

Squeeze the budget and watch the agent hit the wall mid-diagnosis.

In [ ]:
print(run_react_agent('Why is CELL-031A underperforming this morning?', max_steps=2))

> It stops without concluding. That is the *correct* behaviour — an agent that knows it ran out
> of budget is far safer than one that keeps going.

> Without the guard, a flapping cell (PRB alternating 40% / 92%) makes the agent re-query forever.
> It never errors. It just quietly spends money until someone reads the invoice.

---
## Your turn

1. Ask about **CELL-022A**, which is healthy. Does the agent correctly conclude nothing is wrong?
2. Remove `lookup_topology` from the tool list and re-run. Watch it blame the cell it was given.
3. Set `max_steps=10` — does it use the extra budget, or stop when it has enough?

Number 2 is the important one. It is the failure the whole capstone is built to prevent.

In [ ]:
# Your turn
print(run_react_agent('Is there anything wrong with CELL-022A?'))

---
**Next — and this is where the notebooks stop.**

From Module 6 on we are writing files, not cells. Not because notebooks stopped working, but
because the lesson changes: tool schemas, a packaged Skill on disk, an MCP server running as a
process, an eval suite in CI. Those *are* files, and putting them in a notebook would teach the
opposite of the point.

Clone the repo locally and open Module 6 in your editor:

```bash
git clone https://github.com/telcobytes/netops-genai-course.git
cd netops-genai-course && pip install -r requirements.txt
python module06-noc-assistant/noc_assistant.py
```

You have outgrown the notebook. That is progress, not friction.